In [27]:
from pathlib import Path
import pandas as pd

def lectura_solicitudes():
    current_dir = Path.cwd()

    for directory in [current_dir, *current_dir.parents]:
        solicitudes_path = directory / "data" / "solicitudes_de_credito.csv.gz"

        if solicitudes_path.exists():
            return pd.read_csv(
                solicitudes_path,
                sep=None,
                engine="python",
                compression="gzip"
            )

    raise FileNotFoundError(
        "No se encontró data/solicitudes_de_credito.csv.gz"
    )

In [28]:
df=lectura_solicitudes()

In [29]:
def seleccionar_columnas(df):
    columnas = [
        "sexo",
        "tipo_de_emprendimiento",
        "idea_negocio",
        "barrio",
        "estrato",
        "comuna_ciudadano",
        "fecha_de_beneficio",
        "monto_del_credito",
        "línea_credito"
    ]

    df.columns = df.columns.str.strip()

    return df[columnas].copy()

selected_df = seleccionar_columnas(df)

In [30]:
selected_df

,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,masculino,servicio,peluqueria,santa cruz,1,2.0,28/03/2017,3500000,microempresarial
1,femenino,industria,fabrica de,robledo,3,NaN,2018/07/19,7800000,microempresarial
2,masculino,comercio,fabrica de,la milagrosa,3,9.0,22/08/2018,7000000,microempresarial
3,masculino,industria,fabrica de,manrique central no. 2,2,3.0,15/02/2016,2000000,microempresarial
4,femenino,industria,fabrica de,andaluc¿a,2,NaN,6/02/2016,6800000,microempresarial
...,...,...,...,...,...,...,...,...,...
11152,femenino,comercio,granero,BARRIO CAICEDO,3,9.0,26/09/2016,6000000,microempresarial
11153,femenino,comercio,comidas rapidas,cabecera san cristobal,2,60.0,22/02/2018,6000000,microempresarial
11154,masculino,industria,fabrica de,campo valdes no. 1,2,4.0,4/05/2018,1800000,MICROEMPRESARIAL
11155,femenino,comercio,comidas rapidas,20 de julio,1,13.0,2017/10/04,1800000,microempresarial


In [31]:
def limpiar_textos(df):
    columnas_texto = [
        "sexo",
        "tipo_de_emprendimiento",
        "idea_negocio",
        "barrio",
        "comuna_ciudadano",
        "línea_credito"
    ]

    for columna in columnas_texto:
        df[columna] = (
            df[columna]
            .astype("string")
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

        df[columna] = df[columna].replace("", pd.NA)

    return df

In [32]:
limpio_df = limpiar_textos(selected_df)

In [33]:
limpio_df

,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,masculino,servicio,peluqueria,santa cruz,1,2.0,28/03/2017,3500000,microempresarial
1,femenino,industria,fabrica de,robledo,3,<NA>,2018/07/19,7800000,microempresarial
2,masculino,comercio,fabrica de,la milagrosa,3,9.0,22/08/2018,7000000,microempresarial
3,masculino,industria,fabrica de,manrique central no. 2,2,3.0,15/02/2016,2000000,microempresarial
4,femenino,industria,fabrica de,andaluc¿a,2,<NA>,6/02/2016,6800000,microempresarial
...,...,...,...,...,...,...,...,...,...
11152,femenino,comercio,granero,barrio caicedo,3,9.0,26/09/2016,6000000,microempresarial
11153,femenino,comercio,comidas rapidas,cabecera san cristobal,2,60.0,22/02/2018,6000000,microempresarial
11154,masculino,industria,fabrica de,campo valdes no. 1,2,4.0,4/05/2018,1800000,microempresarial
11155,femenino,comercio,comidas rapidas,20 de julio,1,13.0,2017/10/04,1800000,microempresarial


In [34]:

def limpiar_numeros(df):
    columnas_numericas = ["estrato", "monto_del_credito"]

    for columna in columnas_numericas:
        valores = (
            df[columna]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"[^\d]", "", regex=True)
        )

        df[columna] = pd.to_numeric(valores, errors="coerce")

    return df

numeros_limpios_df = limpiar_numeros(limpio_df)
numeros_limpios_df

,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,masculino,servicio,peluqueria,santa cruz,1,2.0,28/03/2017,3500000,microempresarial
1,femenino,industria,fabrica de,robledo,3,<NA>,2018/07/19,7800000,microempresarial
2,masculino,comercio,fabrica de,la milagrosa,3,9.0,22/08/2018,7000000,microempresarial
3,masculino,industria,fabrica de,manrique central no. 2,2,3.0,15/02/2016,2000000,microempresarial
4,femenino,industria,fabrica de,andaluc¿a,2,<NA>,6/02/2016,6800000,microempresarial
...,...,...,...,...,...,...,...,...,...
11152,femenino,comercio,granero,barrio caicedo,3,9.0,26/09/2016,6000000,microempresarial
11153,femenino,comercio,comidas rapidas,cabecera san cristobal,2,60.0,22/02/2018,6000000,microempresarial
11154,masculino,industria,fabrica de,campo valdes no. 1,2,4.0,4/05/2018,1800000,microempresarial
11155,femenino,comercio,comidas rapidas,20 de julio,1,13.0,2017/10/04,1800000,microempresarial


In [35]:
def limpiar_fechas(df):
    df["fecha_de_beneficio"] = pd.to_datetime(
        df["fecha_de_beneficio"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    return df

fechas_limpias_df = limpiar_fechas(numeros_limpios_df)
fechas_limpias_df

,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,masculino,servicio,peluqueria,santa cruz,1,2.0,2017-03-28,3500000,microempresarial
1,femenino,industria,fabrica de,robledo,3,<NA>,2018-07-19,7800000,microempresarial
2,masculino,comercio,fabrica de,la milagrosa,3,9.0,2018-08-22,7000000,microempresarial
3,masculino,industria,fabrica de,manrique central no. 2,2,3.0,2016-02-15,2000000,microempresarial
4,femenino,industria,fabrica de,andaluc¿a,2,<NA>,2016-02-06,6800000,microempresarial
...,...,...,...,...,...,...,...,...,...
11152,femenino,comercio,granero,barrio caicedo,3,9.0,2016-09-26,6000000,microempresarial
11153,femenino,comercio,comidas rapidas,cabecera san cristobal,2,60.0,2018-02-22,6000000,microempresarial
11154,masculino,industria,fabrica de,campo valdes no. 1,2,4.0,2018-05-04,1800000,microempresarial
11155,femenino,comercio,comidas rapidas,20 de julio,1,13.0,2017-10-04,1800000,microempresarial


In [36]:
def eliminar_incompletos(df):
    columnas_obligatorias = [
        "sexo",
        "tipo_de_emprendimiento",
        "idea_negocio",
        "barrio",
        "estrato",
        "fecha_de_beneficio",
        "monto_del_credito",
        "línea_credito"
    ]

    return df.dropna(subset=columnas_obligatorias)

incompletos_eliminados_df = eliminar_incompletos(fechas_limpias_df)
incompletos_eliminados_df

,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,masculino,servicio,peluqueria,santa cruz,1,2.0,2017-03-28,3500000,microempresarial
1,femenino,industria,fabrica de,robledo,3,<NA>,2018-07-19,7800000,microempresarial
2,masculino,comercio,fabrica de,la milagrosa,3,9.0,2018-08-22,7000000,microempresarial
3,masculino,industria,fabrica de,manrique central no. 2,2,3.0,2016-02-15,2000000,microempresarial
4,femenino,industria,fabrica de,andaluc¿a,2,<NA>,2016-02-06,6800000,microempresarial
...,...,...,...,...,...,...,...,...,...
11152,femenino,comercio,granero,barrio caicedo,3,9.0,2016-09-26,6000000,microempresarial
11153,femenino,comercio,comidas rapidas,cabecera san cristobal,2,60.0,2018-02-22,6000000,microempresarial
11154,masculino,industria,fabrica de,campo valdes no. 1,2,4.0,2018-05-04,1800000,microempresarial
11155,femenino,comercio,comidas rapidas,20 de julio,1,13.0,2017-10-04,1800000,microempresarial


In [37]:
def limpiar_guiones_y_espacios(df):
    columnas = ["barrio", "idea_negocio"]

    for columna in columnas:
        df[columna] = (
            df[columna]
            .str.replace("-", " ", regex=False)
            .str.rstrip()
        )

    return df

limpios_guiones_y_espacios_df = limpiar_guiones_y_espacios(incompletos_eliminados_df)

/var/folders/fb/5k8hlzl50bj9rcy1q1k5l6bc0000gn/T/ipykernel_5532/1379036802.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[columna] = (


In [42]:
def limpiar_tildes(df):
    columnas = ["barrio", "idea_negocio"]

    for columna in columnas:
        df[columna] = (
            df[columna]
            .str.normalize("NFKD")
            .str.encode("ascii", errors="ignore")
            .str.decode("utf-8")
        )

    return df


limpios_tildes_df = limpiar_tildes(limpios_guiones_y_espacios_df)


/var/folders/fb/5k8hlzl50bj9rcy1q1k5l6bc0000gn/T/ipykernel_5532/3882012436.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[columna] = (


In [39]:
limpios_guiones_y_espacios_df

,sexo,tipo_de_emprendimiento,idea_negocio,barrio,estrato,comuna_ciudadano,fecha_de_beneficio,monto_del_credito,línea_credito
0,masculino,servicio,peluqueria,santa cruz,1,2.0,2017-03-28,3500000,microempresarial
1,femenino,industria,fabrica de,robledo,3,<NA>,2018-07-19,7800000,microempresarial
2,masculino,comercio,fabrica de,la milagrosa,3,9.0,2018-08-22,7000000,microempresarial
3,masculino,industria,fabrica de,manrique central no. 2,2,3.0,2016-02-15,2000000,microempresarial
4,femenino,industria,fabrica de,andaluc¿a,2,<NA>,2016-02-06,6800000,microempresarial
...,...,...,...,...,...,...,...,...,...
11152,femenino,comercio,granero,barrio caicedo,3,9.0,2016-09-26,6000000,microempresarial
11153,femenino,comercio,comidas rapidas,cabecera san cristobal,2,60.0,2018-02-22,6000000,microempresarial
11154,masculino,industria,fabrica de,campo valdes no. 1,2,4.0,2018-05-04,1800000,microempresarial
11155,femenino,comercio,comidas rapidas,20 de julio,1,13.0,2017-10-04,1800000,microempresarial


In [43]:

#línea_credito
#barrio
#idea_negocio
series = limpios_guiones_y_espacios_df["barrio"]
series = series.sort_values()
series = series.drop_duplicates()
series

3211          13 de noviembre
10452             20 de julio
8090              aguas frias
9222     alejandro echavarria
2169            alfonso lopez
                 ...         
2567        vrda. san antonio
1207      vrda. san cristobal
7627        vrda. santa elena
598                yarumalito
1526       zona expansion san
Name: barrio, Length: 232, dtype: object